# DDR ground-truth lesion probe on the EyePACS-adapted arms: P-EP, E1-EP (and E2-EP only if eligible) — record §68, §69.2

**Nothing runs unless its switch in cell [2] is `True` and its confirmation phrase is typed.** With the defaults the
notebook only shows which checkpoints exist and the state of the E2-EP gate.

The probe is the one of record §68, unchanged (the code is `ddr_probe`'s): the pinned 756-image DDR set (383 / 148 / 225),
expert masks reduced to 16×16 any-pixel cell targets, the same fresh 1×1 probe on the frozen 16×16×768 feature map,
the five-epoch probe (**primary**) and the converged probe (robustness, reported next to it), two test predictions per
probe, and the paired bootstrap over the 225 test images (2,000 resamples, seed 20260927) computed afterwards.

| probe | switch | encoders | condition |
|---|---|---|---|
| **P-EP** and **E1-EP** | `RUN_EP_PROBE` | the pinned BEST checkpoints of both arms, seeds 42 / 123 / 2026 | both arms trained and pinned |
| **E2-EP** | `RUN_E2_EP` | the pinned BEST checkpoints of E2-EP | the gate below is OPEN **and** the phrase is typed |

**Criterion (fixed in §69.2):** on the five-epoch probe, E1-EP − P-EP in mean cell AUROC positive in 3 / 3 seeds and the
95 % interval of the three-seed mean excluding zero. Meeting it makes E2-EP *eligible*; it starts nothing.

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import setup as colab_setup
SETUP_INFO = colab_setup.setup()          # Drive, repo, requirements, env vars. No TJDR / IDRiD / Stage-4 weights needed.
import stage4_v2_setup                    # only for stage_bundle (remount-safe Drive -> local copy)
import glob, json, posixpath
import numpy as np, pandas as pd, tensorflow as tf
import colab_config
import ddr_probe as dp, ddr_probe_ep as dpe, e1_probe as ep
import eyepacs_adaptation_data as ea, eyepacs_adaptation_train as eat, ep_aptos_train as epa
import multiseed_runs as msr, pl_convnext as pl, weighted_corn
import pipeline_v2_config as v2cfg, stage34_cache_v2 as cache
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] SWITCHES -- both OFF. A switch needs its phrase in CONFIRM before anything runs on the GPU. ====
RUN_EP_PROBE = False    # the probe of P-EP and E1-EP (six encoders)
RUN_E2_EP    = False    # the probe of E2-EP -- refused unless the E2-EP gate is open
CONFIRM = {"ep_probe": "", "e2_ep_probe": ""}
PHRASES = {"ep_probe": "PROBE P-EP AND E1-EP", "e2_ep_probe": dpe.E2_EP_PROBE_CONFIRMATION}

def confirmed(step, flag):
    if flag and CONFIRM[step] != PHRASES[step]:
        raise RuntimeError(f"{step}: the switch is True but CONFIRM['{step}'] is not the phrase {PHRASES[step]!r}")
    return bool(flag)
print({"RUN_EP_PROBE": RUN_EP_PROBE, "RUN_E2_EP": RUN_E2_EP})

In [ ]:
# ==== [3] INPUTS AND STATE (reads small files only; nothing is trained or scored) ====
EXP = colab_config.DRIVE.experiments_root
DDR_LESION = posixpath.join(colab_config.DATASET_ROOT, "DDR", "raw", "lesion_segmentation")
DDR_AUDIT = posixpath.join(colab_config.DATASET_ROOT, "DDR", "audit")
PRETRAINED = pl.ensure_pretrained_weights(posixpath.join(EXP, "PL_ConvNeXtPriors", "pretrained_weights"))["local_path"]
assert cache.sha256_file(PRETRAINED) == pl.WEIGHTS_SHA256
ADAPTATION_DIR = eat.run_dir_for(EXP, ea.MANIFEST_SHA256)
OUT_DIR = posixpath.join(EXP, "DDRProbe", "ddr_ground_truth_probe_ep_v1")
BASELINE_DIR = posixpath.join(EXP, "DDRProbe", "ddr_ground_truth_probe_v1")     # the recorded P / E1 / E2 probe (record 70)
ELIGIBILITY = posixpath.join(EXP, epa.EXPERIMENT, epa.ELIGIBILITY_NAME)
print("probe protocol", dpe.PROTOCOL_VERSION, "| DDR manifest", dp.MANIFEST_SHA256[:16], "| probe", ep.PROBE)
print("stopping (validation only)", dp.STOPPING, "| bootstrap", dp.N_BOOT, "resamples, seed", dp.BOOT_SEED)
print("criterion:", dpe.CRITERION)
ADAPTED_SHA = None
if os.path.exists(posixpath.join(ADAPTATION_DIR, eat.FROZEN_NAME)):
    ADAPTED_SHA = eat.read_frozen(ADAPTATION_DIR)[1]["sha256"]
    for _arm in dpe.ARMS:
        print(f"  {_arm:<6}", {s: epa.seed_state(epa.run_dir_for(EXP, _arm, ADAPTED_SHA, s)) for s in dpe.SEEDS})
else:
    print("the EyePACS adaptation checkpoint is not pinned yet -- no EP checkpoint exists")
GATE = dpe.e2_ep_gate_status(ELIGIBILITY)
print("\nE2-EP GATE:", "OPEN (eligible)" if GATE["eligible"] else "CLOSED", "|", GATE["reason"])
if GATE["recorded"]:
    print("  E1-EP - P-EP per seed", GATE["per_seed"], "| mean", GATE["mean"], "| 95% interval", GATE["ci"], "| positive seeds", GATE["positive_seeds"])

In [ ]:
# ==== [4] PROBE OF P-EP AND E1-EP (six frozen encoders; resumable). Runs only when switched on and confirmed. ====
if confirmed("ep_probe", RUN_EP_PROBE):
    assert ADAPTED_SHA is not None, "no pinned adaptation checkpoint"
    assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
    # every checkpoint must be a pinned EP run started from the adapted encoder; its file is re-hashed
    CHECKPOINTS = dpe.write_checkpoint_manifest(posixpath.join(OUT_DIR, "checkpoint_manifest.json"),
                                               dpe.build_checkpoint_manifest(EXP, ADAPTED_SHA, dpe.REQUIRED_ARMS, repo_dir=REPO_DIR))
    for _e in CHECKPOINTS["checkpoints"]:
        print(f"  {_e['arm']:<6} seed {_e['seed']:<5} {_e['model_type']:<3} {_e['sha256']}")
    SUMMARY = dpe.run(EXP, DDR_LESION, DDR_AUDIT, OUT_DIR, CHECKPOINTS, PRETRAINED, repo_dir=REPO_DIR)
    for _key in sorted(SUMMARY["results"], key=lambda k: (int(k.split("seed")[1]), k)):
        _r = SUMMARY["results"][_key]
        print(f"{_key:<18} five-epoch {_r['five_epoch']['test']['scores']['mean']:.4f} | converged {_r['converged']['test']['scores']['mean']:.4f} "
              f"(epoch {_r['behaviour']['converged_epoch']} of {_r['behaviour']['epochs_run']}, {_r['behaviour']['stopped_by']})")
    print("saved ->", OUT_DIR, "\nThe comparison and the criterion are computed afterwards from the saved test logits (dpe.analyse).")
else:
    print("P-EP / E1-EP probe not run (RUN_EP_PROBE = False)")

In [ ]:
# ==== [5] PROBE OF E2-EP -- CONDITIONAL. Shows the gate; refused unless it is open AND the phrase is typed. ====
print("E2-EP GATE:", "OPEN (eligible)" if GATE["eligible"] else "CLOSED", "|", GATE["reason"])
if confirmed("e2_ep_probe", RUN_E2_EP):
    assert ADAPTED_SHA is not None, "no pinned adaptation checkpoint"
    epa.require_e2_ep_eligibility(ELIGIBILITY)                              # raises unless the recorded criterion was met
    E2_OUT_DIR = OUT_DIR + "_with_e2_ep"                                    # its own directory: the P-EP / E1-EP run is not touched
    E2_CHECKPOINTS = dpe.write_checkpoint_manifest(posixpath.join(E2_OUT_DIR, "checkpoint_manifest.json"),
                                                  dpe.build_checkpoint_manifest(EXP, ADAPTED_SHA, tuple(dpe.ARMS), repo_dir=REPO_DIR))
    E2_SUMMARY = dpe.run(EXP, DDR_LESION, DDR_AUDIT, E2_OUT_DIR, E2_CHECKPOINTS, PRETRAINED, repo_dir=REPO_DIR,
                         eligibility=ELIGIBILITY, e2_ep_confirmation=CONFIRM["e2_ep_probe"])
    print("saved ->", E2_OUT_DIR)
else:
    print("E2-EP probe not run (RUN_E2_EP = False)")